In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "01-ai/Yi-1.5-6B-Chat"

# Cấu hình nén 4-bit tối ưu cho GPU T4 của Kaggle
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("--- Đang tải mô hình Yi-1.5-6B-Chat (Mở hoàn toàn) ---")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
print("=> Đã tải xong Yi-1.5-6B-Chat thành công và sẵn sàng!")

In [ ]:
import pandas as pd

# Đường dẫn file csv của bạn
file_path = '/kaggle/input/datasets/richarddoan/input-data/benchmark_descriptions_teacher.csv' 
df = pd.read_csv(file_path)

print(f"Đã nạp file thành công! Kích thước dữ liệu: {df.shape}")

In [ ]:
import json
import re
from tqdm import tqdm

tien_ich_cols = [
    'hem_xe_hoi', 'gan_cho_sieu_thi', 'gan_truong_hoc', 
    'gan_benh_vien', 'gan_cong_vien_ho_nuoc'
]

def extract_features_yi(description):
    # Cấu hình System Prompt định hướng cho mô hình Châu Á hiểu từ lóng Việt Nam
    system_prompt = "Bạn là một trợ lý AI chuyên trích xuất thông tin bất động sản Việt Nam. Hãy hiểu các từ viết tắt như hxh (hẻm xe hơi), st (siêu thị), bv (bệnh viện), cv (công viên), trg (trường học). Bạn luôn trả về kết quả dưới dạng JSON duy nhất, không giải thích gì thêm."
    
    user_prompt = f"""Đọc đoạn mô tả bất động sản sau và xác định xem các tiện ích xung quanh có được nhắc đến hay không.
Trả về giá trị 1 nếu CÓ nhắc đến (hoặc có ý nghĩa tương đương), và 0 nếu KHÔNG nhắc đến.

BẮT BUỘC TRẢ VỀ ĐÚNG ĐỊNH DẠNG JSON VỚI CÁC KEY SAU:
{{
    "hem_xe_hoi": 0,
    "gan_cho_sieu_thi": 0,
    "gan_truong_hoc": 0,
    "gan_benh_vien": 0,
    "gan_cong_vien_ho_nuoc": 0
}}

Mô tả: "{description}" """

    # Đóng gói tin nhắn theo chuẩn Chat Template của Yi
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs, 
            max_new_tokens=150, 
            do_sample=False # Tắt lấy mẫu ngẫu nhiên (Greedy Decoding) để giữ logic cố định
        )
        
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    
    try:
        # Sử dụng Regex bóc tách chuỗi JSON đề phòng mô hình bao bọc bằng ```json ... ```
        if "{" in response:
            json_clean = re.search(r'\{.*\}', response, re.DOTALL).group()
            return json.loads(json_clean)
        return None
    except Exception as e:
        # Nếu dòng nào lỗi định dạng, điền tạm mặc định bằng 0 để mạch code không bị gãy
        return {col: 0 for col in tien_ich_cols}

# --- TIẾN HÀNH CHẠY QUÉT DỮ LIỆU ---
print("Yi-1.5-6B đang tiến hành quét dữ liệu mô tả...")

for index, row in tqdm(df.iterrows(), total=len(df), desc="Xử lý"):
    description = row['Mô tả']
    
    if pd.isna(description):
        continue
        
    extracted_data = extract_features_yi(description)
    
    if extracted_data:
        for col in tien_ich_cols:
            df.at[index, col] = extracted_data.get(col, 0)

# Lưu kết quả ra file mới
output_file = 'benchmark_ground_truth_yi.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"\n=> Hoàn thành! File kết quả lưu tại: /kaggle/working/{output_file}")